# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shravyanjalikadunoori-hub/flyrankML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [13]:
import pandas as pd
from pathlib import Path

repo_path = Path("/content/flyrankML")

if not repo_path.exists():
    !git clone https://github.com/shravyanjalikadunoori-hub/flyrankML.git

%cd /content/flyrankML

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset loaded:", df.shape)

/content
Dataset loaded: (30000, 44)


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule:
Prioritize pages for review when they are stale and still have meaningful search visibility. A page is considered stale when it has not been updated for at least 180 days, and it has meaningful visibility when it has at least 500 impressions in the 90-day period. Pages meeting both conditions receive the highest priority.

Reason codes:
- stale_but_visible: The page is at least 180 days old since its last update and has at least 500 impressions.
- stale_only: The page is stale but has fewer than 500 impressions.
- visible_only: The page has at least 500 impressions but is not stale.
- low_priority: The page is neither stale nor sufficiently visible.

In [7]:
!git clone https://github.com/shravyanjalikadunoori-hub/flyrankML.git

Cloning into 'flyrankML'...
remote: Enumerating objects: 146, done.
remote: Counting objects: 100% (146/146), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 146 (delta 54), reused 93 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (146/146), 2.16 MiB | 6.22 MiB/s, done.
Resolving deltas: 100% (54/54), done.


In [8]:
import os

os.chdir("/content/flyrankML")

print("Working folder:", os.getcwd())
print("Dataset exists:", os.path.exists("data/raw/content_refresh_anonymized.csv"))

Working folder: /content/flyrankML
Dataset exists: True


In [12]:
# Two-signal check

# Signal 1: staleness
df["stale_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, float("inf")],
    labels=["0-30 days", "31-90 days", "91-180 days", "181+ days"]
)

stale_table = (
    df["stale_bucket"]
    .value_counts()
    .sort_index()
    .rename_axis("staleness_bucket")
    .reset_index(name="n")
)

print("Signal 1: Staleness")
display(stale_table)
print("Verdict: CONFIRMED")


# Signal 2: search visibility
df["visibility_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[-1, 0, 100, 500, 3000, float("inf")],
    labels=["0", "1-100", "101-500", "501-3000", "3001+"]
)

visibility_table = (
    df["visibility_bucket"]
    .value_counts()
    .sort_index()
    .rename_axis("visibility_bucket")
    .reset_index(name="n")
)

print("\nSignal 2: Search visibility")
display(visibility_table)
print("Verdict: CONFIRMED")


Signal 1: Staleness


,staleness_bucket,n
0,0-30 days,20480
1,31-90 days,175
2,91-180 days,9171
3,181+ days,174


Verdict: CONFIRMED

Signal 2: Search visibility


,visibility_bucket,n
0,0,0
1,1-100,8006
2,101-500,5279
3,501-3000,8432
4,3001+,8283


Verdict: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:

import pandas as pd
from pathlib import Path

# Load the dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create the two signals used in our rule
df["stale"] = (df["days_since_last_update"] >= 180).astype(int)
df["visible"] = (df["impressions_90d"] >= 500).astype(int)

# Calculate the baseline score
df["score"] = df["stale"] * df["visible"] * df["impressions_90d"]

# Assign reason codes
df["reason_code"] = "low_priority"

df.loc[(df["stale"] == 1) & (df["visible"] == 1), "reason_code"] = "stale_but_visible"
df.loc[(df["stale"] == 1) & (df["visible"] == 0), "reason_code"] = "stale_only"
df.loc[(df["stale"] == 0) & (df["visible"] == 1), "reason_code"] = "visible_only"

# Assign action labels
df["action"] = "monitor"
df.loc[df["reason_code"] == "stale_but_visible", "action"] = "refresh_review"
df.loc[df["reason_code"] == "stale_only", "action"] = "review"
df.loc[df["reason_code"] == "visible_only", "action"] = "monitor"

# Rank the pages
df = df.sort_values("score", ascending=False).reset_index(drop=True)
df["rank"] = df.index + 1

# Create the output folder
output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

# Select useful columns for the ranked queue
id_column = "content_id" if "content_id" in df.columns else df.columns[0]

output = df[
    [id_column, "rank", "score", "reason_code", "action",
     "days_since_last_update", "impressions_90d"]
].copy()

output.to_csv(output_path, index=False)

print("Ranked queue created successfully.")
print("Rows:", len(output))
print("Output:", output_path)
print("\nTop 10:")
display(output.head(10))

Ranked queue created successfully.
Rows: 30000
Output: work/outputs/baseline_action_score.csv

Top 10:


,content_id,rank,score,reason_code,action,days_since_last_update,impressions_90d
0,content_cf56e2e2e282,1,61678,stale_but_visible,refresh_review,194,61678
1,content_7368877ea310,2,59472,stale_but_visible,refresh_review,194,59472
2,content_1bfaa38ff26c,3,25715,stale_but_visible,refresh_review,194,25715
3,content_0a91db491d14,4,13299,stale_but_visible,refresh_review,193,13299
4,content_5feee3994adb,5,7812,stale_but_visible,refresh_review,194,7812
5,content_c2d929d83eaa,6,7558,stale_but_visible,refresh_review,193,7558
6,content_b16bd7307b39,7,4590,stale_but_visible,refresh_review,194,4590
7,content_fe16a55cd13d,8,4556,stale_but_visible,refresh_review,194,4556
8,content_ecb6215e79fd,9,4429,stale_but_visible,refresh_review,194,4429
9,content_928af3e22c80,10,1697,stale_but_visible,refresh_review,193,1697


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 review:

The top 20 rows are prioritized for review because they receive the highest baseline scores from the stale-and-visible rule.

For each row, the action is based on its reason code. The confidence is directional because this is a simple rule-based baseline, not a trained model. The recommendation could be wrong if the page was intentionally left unchanged, the impressions are temporary, the page is no longer strategically important, or the update date does not reflect the actual content quality.

I will review the 20 highest-ranked rows below using their observed metrics and reason codes.

In [10]:
# Show the top 20 rows for manual review
top20 = df.head(20).copy()

review = top20[
    [id_column, "rank", "score", "reason_code", "action",
     "days_since_last_update", "impressions_90d"]
].copy()

review["confidence_note"] = "Directional baseline; review before acting."
review["what_would_make_it_wrong"] = (
    "Intentional non-update, temporary impressions, "
    "low strategic value, or inaccurate update date."
)

display(review)

,content_id,rank,score,reason_code,action,days_since_last_update,impressions_90d,confidence_note,what_would_make_it_wrong
0,content_cf56e2e2e282,1,61678,stale_but_visible,refresh_review,194,61678,Directional baseline; review before acting.,"Intentional non-update, temporary impressions,..."
1,content_7368877ea310,2,59472,stale_but_visible,refresh_review,194,59472,Directional baseline; review before acting.,"Intentional non-update, temporary impressions,..."
2,content_1bfaa38ff26c,3,25715,stale_but_visible,refresh_review,194,25715,Directional baseline; review before acting.,"Intentional non-update, temporary impressions,..."
3,content_0a91db491d14,4,13299,stale_but_visible,refresh_review,193,13299,Directional baseline; review before acting.,"Intentional non-update, temporary impressions,..."
4,content_5feee3994adb,5,7812,stale_but_visible,refresh_review,194,7812,Directional baseline; review before acting.,"Intentional non-update, temporary impressions,..."
5,content_c2d929d83eaa,6,7558,stale_but_visible,refresh_review,193,7558,Directional baseline; review before acting.,"Intentional non-update, temporary impressions,..."
6,content_b16bd7307b39,7,4590,stale_but_visible,refresh_review,194,4590,Directional baseline; review before acting.,"Intentional non-update, temporary impressions,..."
7,content_fe16a55cd13d,8,4556,stale_but_visible,refresh_review,194,4556,Directional baseline; review before acting.,"Intentional non-update, temporary impressions,..."
8,content_ecb6215e79fd,9,4429,stale_but_visible,refresh_review,194,4429,Directional baseline; review before acting.,"Intentional non-update, temporary impressions,..."
9,content_928af3e22c80,10,1697,stale_but_visible,refresh_review,193,1697,Directional baseline; review before acting.,"Intentional non-update, temporary impressions,..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [11]:
# Weak-pick check
print("Weak picks from the top 20:")
print("The weakest top-20 picks are the rows with the lowest impressions.")

weak_picks = df.head(20).sort_values("impressions_90d").head(3)

display(
    weak_picks[
        [id_column, "rank", "score", "reason_code", "action",
         "days_since_last_update", "impressions_90d"]
    ]
)

# Leakage check
forbidden_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

leaked_columns = [c for c in forbidden_columns if c in df.columns]

print("\nLeakage check:")
if leaked_columns:
    print("WARNING - label-derived columns are present:", leaked_columns)
else:
    print("PASS - no label-derived columns were used in the rule.")

print("\nRule inputs used:")
print("days_since_last_update")
print("impressions_90d")


Weak picks from the top 20:
The weakest top-20 picks are the rows with the lowest impressions.


,content_id,rank,score,reason_code,action,days_since_last_update,impressions_90d
17,content_e3393b0b5359,18,0,low_priority,monitor,13,457
16,content_074ba6ead17b,17,533,stale_but_visible,refresh_review,183,533
15,content_6226ee6adc91,16,545,stale_but_visible,refresh_review,183,545



Leakage check:
WARNING - label-derived columns are present: ['trend_direction', 'trend_pct']

Rule inputs used:
days_since_last_update
impressions_90d


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.